In [ ]:
# ------------------------------------------------------------
# C3. Diagnosis — RiverSP Reach-ID Correspondence (anti-adjacent, exact-first)
# Purpose:
#   For each station-linked reach (PRD/SWORD reach_id), find the same physical
#   reach across ALL RiverSP reach shapefiles (C & D), avoiding adjacency
#   mismatches. Report per-station ID stability and a C↔D crosswalk.
#
# Version: 2.0 — 2025-09-03
# Author : ChatGPT (GPT-5 Thinking)
#
# Outputs (BASE/output/diagnosis):
#   - overlap_matches.csv        (one best match per station×file, with diagnostics)
#   - per_station_summary.csv    (stability/versions/PRD agreement)
#   - comparison.gpkg            (layers: station_ref, matched_reaches)
# ------------------------------------------------------------

In [58]:
# =========================
# Block 0 — Imports
# =========================
import os, glob, math
from pathlib import Path
from io import StringIO

import pandas as pd
import geopandas as gpd
import fiona
from shapely.geometry import LineString, Point, Polygon, box
from shapely.ops import unary_union
from shapely import make_valid
from tqdm import tqdm


In [59]:
# =========================
# Block 1 — User Settings
# =========================
BASE = Path(r"C:\Users\ibana\Desktop\JRC_Tanganica\GIS_Intermediate\Intermediate_files\RiverDischarge")
PATH_STATIONS = BASE / r"input\Stations\PatriciaTEAMS\Round_3_Process\All_stations_Rwanda_WaterPortal.shp"  # not used here, but kept for context
PATH_SWOT_BASE = BASE / r"input\SWOT_L2_HR_RiverSP"
PATH_SWORD_AF = BASE / r"input\downloaded\DB_SWORD\shp\AF"

# RiverSP subfolders
RIVERSP_REACH_DIRS = [
    PATH_SWOT_BASE / "versionC" / "reach",
    PATH_SWOT_BASE / "versionD" / "reach",
]
RIVERSP_REACH_GLOB = "**/*.shp"   # keep general; we'll filter to reach layers

# PRD/SWORD reach shapefiles pattern
PRD_SEARCH_GLOB = "**/*reach*.shp"

# Station reach list
STATION_CSV = """station_id,reach_id64,reach_id,distance_m,fallback
239001,17299500251,17299500251,58.26918670532611,False
255501,17299700071,17299700071,35.76680008810896,False
257001,17299700091,17299700091,249.44611091688722,False
259501,17299700121,17299700121,107.71410115789716,False
270001,17299900091,17299900091,34.181683140974386,False
295501,17299900191,17299900191,66.45287430540958,False
69,17299900151,17299900151,64.63857091617552,False
70008,17299900131,17299900131,51.310375903608026,False
70012,17299900241,17299900241,70.89321194025763,False
SW10,17299900221,17299900221,82.07019842467254,False
SW3,13286700181,13286700181,62.55170923307122,False
SW8,17299500011,17299500011,41.62541202450227,False
SW9,17299900236,17299900236,202.12904260568354,False
"""

# ---- Matching thresholds (meters) ----
# AOI + prefilters
STATION_PREFILTER_BUFFER_M = 50000   # 50 km (coarse degree bbox around each station)
MAX_SEARCH_BUFFER_M        = 800     # tighter candidate window around the line (meters)

# Overlap/geometry scoring (strict)
OVERLAP_BUFFER_M           = 100
MIN_OVERLAP_RATIO_BASE     = 0.55     # baseline ref-only overlap (kept for score calc)
MAX_HAUSDORFF_M            = 700

# Anti-adjacency guards (symmetric + endpoints)
SYMMETRIC_OVERLAP_MIN      = 0.65     # min overlap fraction wrt BOTH lines
ENDPOINT_TOL_M             = 500      # both endpoints must align within this (best pairing)

# Policy toggles
PREFER_EXACT_IF_PRESENT              = True
EXCLUDE_ADJACENT_IF_EXACT_PRESENT    = True   # needs PRD neighbor fields (if present)
BASIN_PREFIX_FILTER                  = True   # keep only candidates sharing CBBBBB prefix

# CRS choice for accurate buffering (auto UTM per station)
TARGET_METRIC_CRS = None

# Version hints
VERSION_HINTS = {"versionC":"C","versionD":"D","_CRID-C":"C","_CRID-D":"D","_verC":"C","_verD":"D"}

# Output
OUT_DIR = BASE / "output" / "diagnosis"
GPKG_PATH = OUT_DIR / "comparison.gpkg"
CSV_MATCHES_PATH = OUT_DIR / "overlap_matches.csv"
CSV_SUMMARY_PATH = OUT_DIR / "per_station_summary.csv"
EXPORT_CSV = True
EXPORT_GPKG = True
DEBUG = True

In [60]:
# =========================
# Block 2 — Helpers
# =========================
def log(msg): 
    if DEBUG: print(msg)

def ensure_dir(p: Path):
    p.parent.mkdir(parents=True, exist_ok=True)

def classify_version_from_path(path: str) -> str:
    p = path.replace("\\","/").lower()
    for k,v in VERSION_HINTS.items():
        if k.lower() in p:
            return v
    if "/versionc/" in p: return "C"
    if "/versiond/" in p: return "D"
    return "unknown"

def list_reach_files(root_dirs, pattern=RIVERSP_REACH_GLOB):
    files = []
    for d in root_dirs:
        if d.exists():
            files.extend(glob.glob(str(d / pattern), recursive=True))
    # keep only files that look like reach layers (name or folder contains 'reach')
    files = [f for f in files if ("reach" in os.path.basename(f).lower() or "/reach/" in f.replace("\\","/").lower())]
    return sorted(set(files))

def safe_validize(geom):
    try:
        return make_valid(geom)
    except Exception:
        try: return geom.buffer(0)
        except Exception: return geom

def best_metric_crs_for_lonlat(lon: float, lat: float) -> str:
    zone = int((lon + 180) // 6) + 1
    return f"EPSG:{32600 + zone if lat >= 0 else 32700 + zone}"

def meters_to_deg(lon_deg: float, lat_deg: float, buffer_m: float):
    # quick & safe for bbox prefilter (not for measurement!)
    dlat = buffer_m / 111320.0
    dlon = buffer_m / (111320.0 * max(0.1, math.cos(math.radians(lat_deg))))
    return dlon, dlat

def overlap_score(ref_line_m, cand_line_m):
    # legacy score for ranking; acceptance uses stricter checks
    ref = safe_validize(ref_line_m)
    cand = safe_validize(cand_line_m)
    inter = ref.intersection(cand.buffer(OVERLAP_BUFFER_M))
    if inter.is_empty:
        overlap_ratio = 0.0
    else:
        overlap_ratio = min(1.0, inter.length / max(1e-6, ref.length))
    try:
        hd = ref.hausdorff_distance(cand)
    except Exception:
        hd = float("inf")
    haus_norm = min(1.0, hd / MAX_HAUSDORFF_M)
    score = 0.6*overlap_ratio + 0.4*(1.0 - haus_norm)
    return score, overlap_ratio, hd

def line_endpoints(line):
    coords = list(line.coords)
    return Point(coords[0]), Point(coords[-1])

def endpoint_pair_metrics(ref_m, cand_m):
    """Return (sum_min, max_d) for the best pairing of endpoints."""
    r0, r1 = line_endpoints(ref_m)
    c0, c1 = line_endpoints(cand_m)
    a0, a1 = r0.distance(c0), r1.distance(c1)
    b0, b1 = r0.distance(c1), r1.distance(c0)
    sumA, maxA = a0 + a1, max(a0, a1)
    sumB, maxB = b0 + b1, max(b0, b1)
    return (sumA, maxA) if sumA <= sumB else (sumB, maxB)

def symmetric_overlap_metrics(ref_m, cand_m, buf_m=OVERLAP_BUFFER_M):
    inter = ref_m.intersection(cand_m.buffer(buf_m))
    if inter.is_empty:
        return 0.0, 0.0, 0.0
    inter_len = inter.length
    ol_ref  = inter_len / max(1e-6, ref_m.length)
    ol_cand = inter_len / max(1e-6, cand_m.length)
    ol_min  = min(ol_ref, ol_cand)
    return ol_ref, ol_cand, ol_min

def accept_exact(ref_m, cand_m):
    sc, _, hd = overlap_score(ref_m, cand_m)
    ol_ref, ol_cand, ol_min = symmetric_overlap_metrics(ref_m, cand_m)
    _, ep_max = endpoint_pair_metrics(ref_m, cand_m)
    ok = (ol_min >= SYMMETRIC_OVERLAP_MIN) and (ep_max <= ENDPOINT_TOL_M) and (hd <= MAX_HAUSDORFF_M)
    return ok, {"score": sc, "overlap_ref": ol_ref, "overlap_cand": ol_cand,
                "overlap_min": ol_min, "hausdorff_m": hd, "endpoint_d_max": ep_max}

def accept_nonexact(ref_m, cand_m):
    sc, _, hd = overlap_score(ref_m, cand_m)
    ol_ref, ol_cand, ol_min = symmetric_overlap_metrics(ref_m, cand_m)
    _, ep_max = endpoint_pair_metrics(ref_m, cand_m)
    ok = (ol_min >= max(SYMMETRIC_OVERLAP_MIN, 0.70)) and (ep_max <= min(ENDPOINT_TOL_M, 400)) and (hd <= MAX_HAUSDORFF_M)
    return ok, {"score": sc, "overlap_ref": ol_ref, "overlap_cand": ol_cand,
                "overlap_min": ol_min, "hausdorff_m": hd, "endpoint_d_max": ep_max}

def basin_prefix(rid: str) -> str:
    return str(rid)[:6] if isinstance(rid, str) else str(rid)[:6]

In [61]:
# =========================
# Block 3 — Load Stations & PRD (and neighbor map if available)
# =========================
OUT_DIR.mkdir(parents=True, exist_ok=True)

stations_df = pd.read_csv(StringIO(STATION_CSV))
stations_df["station_id"] = stations_df["station_id"].astype(str)
stations_df["reach_id"]   = stations_df["reach_id"].astype(str)

# Load PRD/SWORD reference
prd_files = sorted(glob.glob(str(PATH_SWORD_AF / PRD_SEARCH_GLOB), recursive=True))
g_prd_list = []
for pf in prd_files:
    try:
        g = gpd.read_file(pf)
        if "reach_id" in g.columns:
            g["reach_id"] = g["reach_id"].astype(str)
            if g.crs is None: g = g.set_crs(4326)
            g_prd_list.append(g)
    except Exception as e:
        log(f"[WARN] PRD read fail {pf}: {e}")

g_prd = None
if g_prd_list:
    # Keep only reach geometry + potential neighbor fields
    keep_cols = ["reach_id","geometry"]
    # Try to carry neighbor fields if present (optional)
    maybe_neighbors = [c for c in set().union(*(set(df.columns) for df in g_prd_list))
                       if c.lower() in {"rch_id_up","rch_id_dn","up_reach","dn_reach","up_id","dn_id","upriver_id","downriver_id"}]
    keep_cols += [c for c in maybe_neighbors if c in set().union(*(set(df.columns) for df in g_prd_list))]
    g_prd = pd.concat([df[[c for c in keep_cols if c in df.columns]] for df in g_prd_list], ignore_index=True)
    g_prd = gpd.GeoDataFrame(g_prd, geometry="geometry", crs=g_prd_list[0].crs)
    g_prd = g_prd.drop_duplicates(subset=["reach_id"])
    log(f"PRD/SWORD reaches loaded: {len(g_prd)}")
else:
    log("[INFO] No PRD/SWORD reach layer found — fallback to RiverSP for station refs if needed.")

# Build PRD neighbor map (optional anti-adjacent when exact present)
PRD_NEIGHBORS = {}
if g_prd is not None:
    up_fields = [c for c in g_prd.columns if c.lower() in {"rch_id_up","up_reach","up_id","upriver_id"}]
    dn_fields = [c for c in g_prd.columns if c.lower() in {"rch_id_dn","dn_reach","dn_id","downriver_id"}]
    for r in g_prd.itertuples():
        nbrs = set()
        for f in up_fields + dn_fields:
            if f in g_prd.columns:
                v = getattr(r, f, None)
                # handle array-like or scalar
                try:
                    if isinstance(v, (list, tuple, set)):
                        nbrs.update(str(x) for x in v if pd.notna(x))
                    else:
                        s = str(v)
                        if s and s.lower() not in {"nan","none",""}:
                            nbrs.add(s)
                except Exception:
                    pass
        PRD_NEIGHBORS[str(r.reach_id)] = nbrs

# Build initial station reference lines from PRD
ref_rows = []
for rec in stations_df[["station_id","reach_id"]].drop_duplicates().to_dict("records"):
    sid, rid = rec["station_id"], rec["reach_id"]
    geom = None
    has_prd = False
    if g_prd is not None:
        hit = g_prd[g_prd["reach_id"] == rid]
        if len(hit) >= 1:
            geom = hit.iloc[0].geometry
            has_prd = True
    ref_rows.append({"station_id": sid, "station_reach_id_ref": rid, "has_prd": has_prd, "geometry": geom})

g_station = gpd.GeoDataFrame(ref_rows, geometry="geometry", crs="EPSG:4326")


PRD/SWORD reaches loaded: 21441


In [62]:
# =========================
# Block 4 — Build AOI from stations (buffer meters → 4326 bbox). Fill gaps later.
# =========================
buffers_4326 = []
# If no PRD geometries at all, fallback AOI to PRD extent (Africa) if available; else skip AOI prefilter.
if g_station["geometry"].notna().any():
    for row in g_station.itertuples():
        geom = row.geometry if row.geometry is not None else None
        if geom is None:
            continue
        centroid = geom.centroid
        metric_crs = TARGET_METRIC_CRS or best_metric_crs_for_lonlat(centroid.x, centroid.y)
        buf = gpd.GeoSeries([geom], crs=4326).to_crs(metric_crs).buffer(STATION_PREFILTER_BUFFER_M).to_crs(4326).iloc[0]
        buffers_4326.append(buf)
elif g_prd is not None and not g_prd.empty:
    # use PRD global extent as AOI
    buffers_4326.append(box(*g_prd.total_bounds))

if buffers_4326:
    aoi_union = unary_union(buffers_4326)
    aoi_bbox_4326 = aoi_union.envelope.bounds  # (minx, miny, maxx, maxy)
    AOI_BOX = box(*aoi_bbox_4326)
    log(f"AOI bbox (deg): {aoi_bbox_4326}")
else:
    aoi_bbox_4326 = None
    AOI_BOX = None
    log("[INFO] AOI prefilter disabled (no station/PRD geometry available).")

AOI bbox (deg): (28.49196087690428, -3.1631417931004275, 31.232620876090323, -0.6169462979534881)


In [63]:
# =========================
# Block 5 — Preselect RiverSP files by AOI bounds
# =========================
reach_files_all = list_reach_files(RIVERSP_REACH_DIRS, pattern=RIVERSP_REACH_GLOB)
reach_files = []
for f in reach_files_all:
    try:
        with fiona.open(f) as src:
            minx, miny, maxx, maxy = src.bounds
        layer_box = box(minx, miny, maxx, maxy)
        if AOI_BOX is None or layer_box.intersects(AOI_BOX):
            reach_files.append(f)
    except Exception as e:
        log(f"[WARN] bounds fail {f}: {e}")
log(f"RiverSP files intersecting AOI: {len(reach_files)} / {len(reach_files_all)}")
if not reach_files:
    raise FileNotFoundError("No RiverSP reach shapefiles intersect AOI. Check paths/AOI.")


RiverSP files intersecting AOI: 826 / 878


In [64]:

# =========================
# Block 6 — Fill missing station refs from first matching RiverSP (AOI-filtered)
# =========================
def read_reach_minimal(path, bbox=None):
    g = gpd.read_file(path, bbox=bbox)
    if "reach_id" not in g.columns:
        return None
    g["reach_id"] = g["reach_id"].astype(str)
    if g.crs is None: g = g.set_crs(4326)
    g["source_file"] = os.path.basename(path)
    g["source_dir"]  = os.path.dirname(path)
    g["version_hint"] = classify_version_from_path(path)
    g["obs_time"] = None
    for tcol in ("time_str","time_tai","time"):
        if tcol in g.columns:
            g["obs_time"] = g[tcol].astype(str)
            break
    return g[["reach_id","geometry","source_file","source_dir","version_hint","obs_time"]]

if g_station["geometry"].isna().any():
    for f in reach_files:
        g = read_reach_minimal(f, bbox=aoi_bbox_4326 if aoi_bbox_4326 else None)
        if g is None or g.empty:
            continue
        for idx, row in g_station[g_station["geometry"].isna()].iterrows():
            rid = row["station_reach_id_ref"]
            hit = g[g["reach_id"] == rid]
            if len(hit):
                g_station.at[idx, "geometry"] = hit.iloc[0].geometry
                g_station.at[idx, "has_prd"]  = False
        if not g_station["geometry"].isna().any():
            break

if g_station["geometry"].isna().any():
    missing = g_station[g_station["geometry"].isna()][["station_id","station_reach_id_ref"]]
    raise RuntimeError(f"No reference geometry found for some stations:\n{missing}")

In [65]:
# =========================
# Block 7 — Matching (exact-first + anti-adjacent)
# =========================
matches = []
log("Scanning filtered RiverSP files…")

# Precompute station coarse degree boxes
station_coarse_boxes = {}
for st in g_station.itertuples():
    geom = st.geometry
    cen = geom.centroid
    dlon, dlat = meters_to_deg(cen.x, cen.y, STATION_PREFILTER_BUFFER_M)
    station_coarse_boxes[st.station_id] = box(cen.x - dlon, cen.y - dlat, cen.x + dlon, cen.y + dlat)

for f in tqdm(reach_files):
    g = read_reach_minimal(f, bbox=aoi_bbox_4326 if aoi_bbox_4326 else None)
    if g is None or g.empty: 
        continue
    g = gpd.GeoDataFrame(g, geometry="geometry", crs=g.crs)
    g_sidx = g.sindex

    for st in g_station.itertuples():
        exact_rid = str(st.station_reach_id_ref)
        # coarse prefilter
        box_deg = station_coarse_boxes[st.station_id]
        try:
            idxs = list(g_sidx.query(box_deg, predicate="intersects"))
        except Exception:
            idxs = list(g_sidx.intersection(box_deg.bounds))
        if not idxs:
            continue
        cand_coarse = g.iloc[idxs].copy()
        if cand_coarse.empty:
            continue

        # optional basin-prefix narrowing
        if BASIN_PREFIX_FILTER:
            bp_ref = basin_prefix(exact_rid)
            cand_coarse = cand_coarse[cand_coarse["reach_id"].astype(str).str[:6] == bp_ref]
            if cand_coarse.empty:
                continue

        # project to local metric CRS
        ref_geom_4326 = safe_validize(st.geometry)
        metric_crs = TARGET_METRIC_CRS or best_metric_crs_for_lonlat(ref_geom_4326.centroid.x, ref_geom_4326.centroid.y)
        ref_m = gpd.GeoSeries([ref_geom_4326], crs=4326).to_crs(metric_crs).iloc[0]
        cand_m = cand_coarse.to_crs(metric_crs)

        # tight buffer
        try:
            idxs2 = list(cand_m.sindex.query(ref_m.buffer(MAX_SEARCH_BUFFER_M), predicate="intersects"))
        except Exception:
            idxs2 = list(cand_m.sindex.intersection(ref_m.buffer(MAX_SEARCH_BUFFER_M).bounds))
        if not idxs2:
            continue
        cand_near = cand_m.iloc[idxs2].copy()
        if cand_near.empty:
            continue

        # score candidates (exact-first logic)
        cand_rows = []
        exact_present_in_file = False
        for c in cand_near.itertuples():
            rid = str(c.reach_id)
            if rid == exact_rid:
                exact_present_in_file = True
                ok, metrics = accept_exact(ref_m, c.geometry)
                if ok:
                    cand_rows.append({**metrics, "rid": rid, "type": "exact",
                                      "source_file": c.source_file, "source_dir": c.source_dir,
                                      "version_hint": c.version_hint, "obs_time": c.obs_time,
                                      "reason": "exact_ok"})
                else:
                    # record failure reason for debugging
                    ok2, m2 = accept_nonexact(ref_m, c.geometry)  # check how close it was
                    cand_rows.append({**m2, "rid": rid, "type": "exact_failed",
                                      "source_file": c.source_file, "source_dir": c.source_dir,
                                      "version_hint": c.version_hint, "obs_time": c.obs_time,
                                      "reason": "exact_failed_thresholds"})
            else:
                ok, metrics = accept_nonexact(ref_m, c.geometry)
                if ok:
                    # optional: drop immediate PRD neighbors when exact exists
                    if EXCLUDE_ADJACENT_IF_EXACT_PRESENT and g_prd is not None and exact_present_in_file:
                        if rid in PRD_NEIGHBORS.get(exact_rid, set()):
                            continue
                    cand_rows.append({**metrics, "rid": rid, "type": "nonexact",
                                      "source_file": c.source_file, "source_dir": c.source_dir,
                                      "version_hint": c.version_hint, "obs_time": c.obs_time,
                                      "reason": "exact_absent_or_nonexact_ok"})

        if not cand_rows:
            continue

        # choose best:
        exact_candidates = [r for r in cand_rows if r["type"] == "exact"]
        if PREFER_EXACT_IF_PRESENT and exact_candidates:
            best = max(exact_candidates, key=lambda r: (r["overlap_min"], -r["endpoint_d_max"], r["score"]))
        else:
            nonexact_ok = [r for r in cand_rows if r["type"] == "nonexact"]
            if nonexact_ok:
                best = max(nonexact_ok, key=lambda r: (r["overlap_min"], -r["endpoint_d_max"], r["score"]))
            else:
                # If only exact_failed entries remain, skip this file; they didn't pass thresholds
                continue

        matches.append({
            "station_id": st.station_id,
            "station_reach_id_ref": exact_rid,
            "ref_has_prd": st.has_prd,
            "matched_reach_id": best["rid"],
            "source_file": best["source_file"],
            "source_dir": best["source_dir"],
            "version_hint": best["version_hint"],
            "obs_time": best["obs_time"],
            "score": best["score"],
            "overlap_ref": best["overlap_ref"],
            "overlap_cand": best["overlap_cand"],
            "overlap_min": best["overlap_min"],
            "hausdorff_m": best["hausdorff_m"],
            "endpoint_d_max": best["endpoint_d_max"],
            "match_type": best["type"],
            "reason": best["reason"],
        })

df_matches = pd.DataFrame(matches)
if df_matches.empty:
    raise RuntimeError("No acceptable matches found — check thresholds or inputs.")

df_matches = df_matches.sort_values(
    ["station_id","station_reach_id_ref","version_hint","source_dir","source_file","overlap_min","score"],
    ascending=[True, True, True, True, True, False, False]
).reset_index(drop=True)


Scanning filtered RiverSP files…


100%|██████████| 826/826 [01:17<00:00, 10.65it/s]﻿


In [66]:
# =========================
# Block 8 — Summary & PRD agreement
# =========================
def versions_presence(vlist):
    s = set(v for v in vlist if v and v != "unknown")
    if s == {"C"}: return "only_C"
    if s == {"D"}: return "only_D"
    if {"C","D"}.issubset(s): return "both"
    return "unknown"

rows = []
for (sid, rid), grp in df_matches.groupby(["station_id","station_reach_id_ref"]):
    ids = grp["matched_reach_id"].dropna().unique().tolist()
    versions = grp["version_hint"].tolist()
    rows.append({
        "station_id": sid,
        "station_reach_id_ref": rid,
        "n_files_matched": grp["source_file"].nunique(),
        "versions_seen": ",".join(sorted(set(v for v in versions if v))),
        "version_presence": versions_presence(versions),
        "distinct_matched_reach_ids": len(ids),
        "matched_reach_ids_sample": ", ".join(ids[:5]),
        "id_stable_across_files": (len(ids) == 1),
    })
df_summary = pd.DataFrame(rows).sort_values(["station_id","station_reach_id_ref"])

# PRD agreement stats
if g_prd is not None:
    prd_ids = set(g_prd["reach_id"].astype(str).unique())
    df_matches["in_prd_same_id"] = df_matches.apply(
        lambda r: (r["matched_reach_id"] == r["station_reach_id_ref"]) and (r["matched_reach_id"] in prd_ids),
        axis=1
    )
    prd_stats = (df_matches.groupby(["station_id","station_reach_id_ref"])["in_prd_same_id"]
                 .agg(["sum","count"]).reset_index())
    prd_stats["pct_same_as_prd"] = (prd_stats["sum"]/prd_stats["count"]).round(3)
    df_summary = df_summary.merge(
        prd_stats[["station_id","station_reach_id_ref","pct_same_as_prd"]],
        on=["station_id","station_reach_id_ref"], how="left"
    )
else:
    df_summary["pct_same_as_prd"] = None

In [67]:
# =========================
# Block 9 — Export (CSV + GPKG, with suffix-safe merge)
# =========================
if EXPORT_CSV:
    ensure_dir(CSV_MATCHES_PATH)
    ensure_dir(CSV_SUMMARY_PATH)
    df_matches.to_csv(CSV_MATCHES_PATH, index=False)
    df_summary.to_csv(CSV_SUMMARY_PATH, index=False)
    log(f"Wrote: {CSV_MATCHES_PATH}")
    log(f"Wrote: {CSV_SUMMARY_PATH}")

if EXPORT_GPKG:
    ensure_dir(GPKG_PATH)
    # station refs
    g_station_out = g_station.copy()
    g_station_out.to_file(GPKG_PATH, layer="station_ref", driver="GPKG")

    # matched geometries: reload per file (AOI-filtered set), suffix control
    agg = []
    for (sdir, sfile) in df_matches[["source_dir","source_file"]].drop_duplicates().itertuples(index=False):
        fpath = os.path.join(sdir, sfile)
        gfile = read_reach_minimal(fpath, bbox=aoi_bbox_4326 if aoi_bbox_4326 else None)
        if gfile is None or gfile.empty: 
            continue

        sub = df_matches[(df_matches["source_dir"]==sdir) & (df_matches["source_file"]==sfile)]
        if sub.empty:
            continue

        # keep best per station in this file
        keep = (sub.sort_values(["station_id","overlap_min","score"], ascending=[True, False, False])
                  .drop_duplicates(subset=["station_id"], keep="first"))

        joined = gfile.merge(
            keep[["station_id","station_reach_id_ref","matched_reach_id",
                  "version_hint","source_file","source_dir","obs_time",
                  "overlap_ref","overlap_cand","overlap_min","hausdorff_m","endpoint_d_max","match_type","reason"]],
            left_on="reach_id", right_on="matched_reach_id", how="inner",
            suffixes=("_src","")   # left gets _src, right keeps clean names
        )
        if len(joined):
            agg.append(joined)

    if agg:
        g_all = gpd.GeoDataFrame(pd.concat(agg, ignore_index=True), geometry="geometry", crs="EPSG:4326")
        wanted = ["station_id","station_reach_id_ref","matched_reach_id",
                  "version_hint","source_file","source_dir","obs_time",
                  "overlap_ref","overlap_cand","overlap_min","hausdorff_m","endpoint_d_max","match_type","reason",
                  "geometry"]
        have = [c for c in wanted if c in g_all.columns]
        g_all = g_all[have]
        g_all.to_file(GPKG_PATH, layer="matched_reaches", driver="GPKG")
        log(f"Wrote: {GPKG_PATH}")
    else:
        log("[INFO] No matched geometries to write to GPKG (check thresholds/AOI).")


Wrote: C:\Users\ibana\Desktop\JRC_Tanganica\GIS_Intermediate\Intermediate_files\RiverDischarge\output\diagnosis\overlap_matches.csv
Wrote: C:\Users\ibana\Desktop\JRC_Tanganica\GIS_Intermediate\Intermediate_files\RiverDischarge\output\diagnosis\per_station_summary.csv
Wrote: C:\Users\ibana\Desktop\JRC_Tanganica\GIS_Intermediate\Intermediate_files\RiverDischarge\output\diagnosis\comparison.gpkg


In [68]:
# =========================
# Block 10 — Console summary & tips
# =========================
print("\n=== Per-station ID stability summary ===")
print(df_summary.to_string(index=False))
print("\nTips:")
print(" - 'both' + id_stable_across_files=True → IDs stable; C vs D diffs are QA/coverage.")
print(" - only_C/only_D + distinct IDs>1 → likely PRD/SWORD ID change across CRIDs.")
print(" - pct_same_as_prd < 1.0 → RiverSP IDs differ from the PRD reference for some files.")
print(" - Inspect 'match_type' and diagnostics in overlap_matches.csv for any nonexact selections.")


=== Per-station ID stability summary ===
station_id station_reach_id_ref  n_files_matched versions_seen version_presence  distinct_matched_reach_ids matched_reach_ids_sample  id_stable_across_files  pct_same_as_prd
    239001          17299500251               71           C,D             both                           2 17299500291, 17299500251                   False            0.127
    255501          17299700071               71           C,D             both                           1              17299700071                    True            1.000
    257001          17299700091               71           C,D             both                           1              17299700091                    True            1.000
    259501          17299700121              147           C,D             both                           1              17299700121                    True            1.000
    270001          17299900091              147           C,D             both         